# `gdrive_fsspec` smoke test

## Example 1: fsspec registry resolves the `gdrive` protocol

In [ ]:
# fsspec's registry maps the "gdrive" protocol to gdrive_fsspec.GoogleDriveFileSystem.
import fsspec
from gdrive_fsspec import GoogleDriveFileSystem

cls = fsspec.get_filesystem_class("gdrive")
assert cls is GoogleDriveFileSystem, cls
assert cls.protocol == "gdrive"
print("gdrive ->", f"{cls.__module__}.{cls.__qualname__}")

## Example 2: anonymous-token construction (offline)

In [ ]:
# token="anon" -> AnonymousCredentials + googleapiclient build("drive", "v3") from
# googleapiclient's static discovery document (no network, no credentials).
from gdrive_fsspec import GoogleDriveFileSystem

fs = GoogleDriveFileSystem(token="anon")
assert fs.root_file_id == "root" and fs.drive is None
assert fs.srv is not None and fs.files is not None  # Drive v3 Resource handles
assert fs.access == "full_control"
assert fs.scopes == ["https://www.googleapis.com/auth/drive"]
print("anon fs ready; scopes:", fs.scopes)

## Example 3: protocol strip/unstrip round trip

In [ ]:
# fsspec path-handling invariant: gdrive:// URLs strip to bare paths and back.
from gdrive_fsspec import GoogleDriveFileSystem

fs = GoogleDriveFileSystem(token="anon")
stripped = GoogleDriveFileSystem._strip_protocol("gdrive://folder/file.txt")
assert stripped == "folder/file.txt", stripped
assert fs.unstrip_protocol(stripped) == "gdrive://folder/file.txt"
assert GoogleDriveFileSystem._strip_protocol("folder/file.txt") == "folder/file.txt"
print("round trip OK:", fs.unstrip_protocol(stripped))

## Example 4: fsspec instance-cache invariant

In [ ]:
# AbstractFileSystem caches instances by storage-options token — identical options
# must return the SAME object; skip_instance_cache opts out.
from gdrive_fsspec import GoogleDriveFileSystem

fs1 = GoogleDriveFileSystem(token="anon")
fs2 = GoogleDriveFileSystem(token="anon")
assert fs1 is fs2
fs3 = GoogleDriveFileSystem(token="anon", skip_instance_cache=True)
assert fs3 is not fs1
GoogleDriveFileSystem.clear_instance_cache()
print("instance cache invariants OK")

## Example 5: documented error paths + module constants

In [ ]:
# Real-user failure paths: an invalid token method and malformed service-account
# creds (google.auth MalformedError is a ValueError subclass).
from gdrive_fsspec import GoogleDriveFileSystem
from gdrive_fsspec.core import DIR_MIME_TYPE, MultipleFilesError, scope_dict

assert issubclass(MultipleFilesError, FileNotFoundError)
assert sorted(scope_dict) == ["full_control", "read_only"]
assert DIR_MIME_TYPE == "application/vnd.google-apps.folder"
try:
    GoogleDriveFileSystem(token="bogus", skip_instance_cache=True)
    raise AssertionError("expected ValueError for invalid token method")
except ValueError as e:
    print("invalid method ->", e)
try:
    GoogleDriveFileSystem(token="service_account", creds={}, skip_instance_cache=True)
    raise AssertionError("expected malformed-creds failure")
except ValueError as e:
    print("malformed creds ->", type(e).__name__)